# 🚀 Google Colab 快速启动指南

本Notebook专为Google Colab优化，自动处理环境配置和GPU设置。

## 第一步：检查GPU并安装依赖

In [ ]:
# 检查GPU可用性
import torch
print("=" * 60)
print("GPU检查")
print("=" * 60)
print(f"CUDA可用: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU设备: {torch.cuda.get_device_name(0)}")
    print(f"CUDA版本: {torch.version.cuda}")
    print(f"显存总量: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("⚠️ 未检测到GPU，请在菜单栏选择: 代码执行程序 -> 更改运行时类型 -> GPU")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"\n使用设备: {device}")

In [ ]:
# 克隆GitHub仓库
!git clone https://github.com/GaoYIZ/PCG_Island.git
%cd PCG_Island

In [ ]:
# 安装依赖
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install -q numpy matplotlib scipy gymnasium opencv-python scikit-image scikit-learn tqdm cma

print("✅ 依赖安装完成！")

In [ ]:
# 验证所有模块导入
import sys
sys.path.append('/content/PCG_Island')

from pcg_generator import PCGIslandGenerator
from structure_evaluator import StructureEvaluator
from vae_model import BetaVAE, HeightmapDataset
from rl_environment import IslandGenerationEnv
from sac_agent import SACAgent, ReplayBuffer
from cmaes_baseline import CMAESOptimizer
from ppo_baseline import PPOAgent
from diversity_analyzer import DiversityAnalyzer

print("✅ 所有模块导入成功！")

## 第二步：快速测试（5分钟）

In [ ]:
# 测试PCG生成
import numpy as np
import matplotlib.pyplot as plt

generator = PCGIslandGenerator(map_size=64)

params = {
    'f': 10, 'A': 1.0, 'N_octaves': 4, 'persistence': 0.5,
    'lacunarity': 2.0, 'seed': 42, 'warp_strength': 0.5,
    'warp_frequency': 2, 'falloff_radius': 32, 'falloff_exponent': 2
}

heightmap = generator.generate_heightmap(params)

# 可视化
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].imshow(heightmap, cmap='terrain')
axes[0].set_title('Heightmap')
plt.colorbar(axes[0].imshow(heightmap, cmap='terrain'), ax=axes[0])

# 3D视图
ax = fig.add_subplot(122, projection='3d')
step = 2
x = np.arange(0, heightmap.shape[0], step)
y = np.arange(0, heightmap.shape[1], step)
X, Y = np.meshgrid(x, y)
Z = heightmap[::step, ::step]
ax.plot_surface(X, Y, Z, cmap='terrain')
ax.set_title('3D Terrain')

plt.tight_layout()
plt.savefig('test_pcg.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"✅ PCG测试成功！高度图范围: [{heightmap.min():.3f}, {heightmap.max():.3f}]")

In [ ]:
# 测试结构评估
evaluator = StructureEvaluator(map_size=64)
metrics = evaluator.evaluate(heightmap)

print("📊 结构评估指标:")
print("=" * 50)
for key, value in metrics.items():
    print(f"{key:25s}: {value:.4f}")

print("\n✅ 结构评估测试成功！")

## 第三步：训练β-VAE（GPU加速，约10-15分钟）

In [ ]:
# 生成小型数据集（Colab免费版建议100-200样本）
print("生成训练数据集...")
n_samples = 100  # Colab免费版建议值
heightmaps = []

for i in range(n_samples):
    params = {
        'f': np.random.uniform(5, 20),
        'A': np.random.uniform(0.5, 1.5),
        'N_octaves': np.random.randint(3, 6),
        'persistence': np.random.uniform(0.3, 0.7),
        'lacunarity': np.random.uniform(1.5, 2.5),
        'seed': i,
        'warp_strength': np.random.uniform(0.2, 0.8),
        'warp_frequency': np.random.uniform(1, 5),
        'falloff_radius': np.random.uniform(20, 40),
        'falloff_exponent': np.random.uniform(1.5, 3)
    }
    heightmap = generator.generate_heightmap(params)
    heightmaps.append(heightmap)

heightmaps = np.array(heightmaps)
print(f"✅ 生成了 {n_samples} 个高度图")

In [ ]:
# 创建数据集
from torch.utils.data import DataLoader

dataset = HeightmapDataset(heightmaps)
dataloader = DataLoader(dataset, batch_size=16, shuffle=True)

# 创建VAE模型
vae = BetaVAE(map_size=64, latent_dim=32, beta=4.0).to(device)

print(f"VAE模型已加载到: {device}")
print(f"参数量: {sum(p.numel() for p in vae.parameters()):,}")

In [ ]:
# 训练VAE
from vae_model import train_vae
import time

print("开始训练β-VAE...")
start_time = time.time()

vae_loss_history = train_vae(
    vae, dataloader, 
    epochs=30,  # Colab建议30-50 epochs
    learning_rate=1e-3, 
    device=device
)

training_time = time.time() - start_time
print(f"\n✅ VAE训练完成！耗时: {training_time:.2f}秒 ({training_time/60:.2f}分钟)")

In [ ]:
# 绘制训练曲线
epochs_list = [h['epoch'] for h in vae_loss_history]
total_losses = [h['total_loss'] for h in vae_loss_history]

plt.figure(figsize=(10, 5))
plt.plot(epochs_list, total_losses, linewidth=2)
plt.xlabel('Epoch')
plt.ylabel('Total Loss')
plt.title('VAE Training Loss (Colab)')
plt.grid(True, alpha=0.3)
plt.savefig('vae_training_colab.png', dpi=150, bbox_inches='tight')
plt.show()

print("✅ 训练曲线已保存！")

## 第四步：SAC强化学习训练（GPU加速，约15-20分钟）

In [ ]:
# 创建环境和SAC智能体
env = IslandGenerationEnv(map_size=64, max_steps=30)

state_dim = env.observation_space.shape[0]
action_dim = env.action_space.shape[0]

agent = SACAgent(
    state_dim=state_dim,
    action_dim=action_dim,
    hidden_dim=256,
    learning_rate=3e-4,
    gamma=0.99,
    tau=0.005,
    alpha=0.2,
    action_range=0.1
).to(device)

print(f"SAC智能体已加载到: {device}")
print(f"状态维度: {state_dim}, 动作维度: {action_dim}")

In [ ]:
# 训练SAC（Colab建议50-100 episodes）
print("开始SAC训练...")
start_time = time.time()

n_episodes = 50  # Colab建议值
batch_size = 64
buffer_capacity = 5000

replay_buffer = ReplayBuffer(capacity=buffer_capacity)

episode_rewards = []
losses_history = []

for episode in range(n_episodes):
    state, _ = env.reset(seed=episode)
    episode_reward = 0
    
    for step in range(env.max_steps):
        action = agent.select_action(state)
        next_state, reward, terminated, truncated, info = env.step(action)
        done = terminated or truncated
        
        replay_buffer.push(state, action, reward, next_state, done)
        
        if len(replay_buffer) >= 500:
            losses = agent.update(replay_buffer, batch_size=batch_size)
            losses_history.append(losses)
        
        episode_reward += reward
        state = next_state
        
        if done:
            break
    
    episode_rewards.append(episode_reward)
    
    if (episode + 1) % 10 == 0:
        avg_reward = np.mean(episode_rewards[-10:])
        print(f"Episode {episode+1}/{n_episodes}, Avg Reward: {avg_reward:.4f}")

training_time = time.time() - start_time
print(f"\n✅ SAC训练完成！耗时: {training_time:.2f}秒 ({training_time/60:.2f}分钟)")

In [ ]:
# 绘制奖励曲线
plt.figure(figsize=(10, 5))
plt.plot(episode_rewards, marker='o', markersize=3, alpha=0.6, label='Raw')

if len(episode_rewards) >= 10:
    moving_avg = np.convolve(episode_rewards, np.ones(10)/10, mode='valid')
    plt.plot(range(9, len(episode_rewards)), moving_avg, 
             linewidth=2, color='red', label='10-Episode MA')

plt.xlabel('Episode')
plt.ylabel('Total Reward')
plt.title('SAC Training Rewards (Colab)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.savefig('sac_training_colab.png', dpi=150, bbox_inches='tight')
plt.show()

print("✅ 训练曲线已保存！")

## 第五步：生成测试岛屿并可视化

In [ ]:
# 使用训练好的策略生成岛屿
print("生成测试岛屿...")
n_test = 10
test_islands = []
test_metrics = []

for i in range(n_test):
    env = IslandGenerationEnv(map_size=64, max_steps=30)
    state, _ = env.reset(seed=i*10)
    
    for step in range(env.max_steps):
        action = agent.select_action(state, evaluate=True)
        state, reward, terminated, truncated, info = env.step(action)
        
        if terminated or truncated:
            break
    
    test_islands.append(info['heightmap'])
    test_metrics.append(info['metrics'])

print(f"✅ 生成了 {n_test} 个测试岛屿")

In [ ]:
# 可视化生成的岛屿网格
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
axes = axes.ravel()

for i in range(n_test):
    im = axes[i].imshow(test_islands[i], cmap='terrain')
    conn = test_metrics[i]['connectivity']
    nav = test_metrics[i]['navigable_ratio']
    axes[i].set_title(f'#{i+1}\nC={conn:.2f}, N={nav:.2f}', fontsize=8)
    axes[i].axis('off')

plt.suptitle('Generated Islands by SAC (Colab)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('generated_islands_colab.png', dpi=150, bbox_inches='tight')
plt.show()

print("✅ 岛屿网格已保存！")

## 第六步：统计结果

In [ ]:
# 统计分析
print("\n📊 测试结果统计:")
print("=" * 60)

metric_keys = list(test_metrics[0].keys())
for key in metric_keys:
    values = [m[key] for m in test_metrics]
    mean_val = np.mean(values)
    std_val = np.std(values)
    print(f"{key:25s}: {mean_val:.4f} ± {std_val:.4f}")

print("\n🎉 Colab测试完成！")
print("\n💡 提示:")
print("- 所有图片已保存到当前目录")
print("- 可以下载图片和模型文件")
print("- 如需完整实验，请运行 full_experiment.ipynb")

## 可选：保存模型到Google Drive

In [ ]:
# 如果需要保存模型到Google Drive，取消下面的注释
# from google.colab import drive
# drive.mount('/content/drive')

# 保存模型
import os
os.makedirs('saved_models', exist_ok=True)

torch.save(vae.state_dict(), 'saved_models/beta_vae.pth')
agent.save('saved_models/sac_agent.pth')

print("✅ 模型已保存到 saved_models/ 目录")

# 如果挂载了Google Drive，可以复制到Drive
# !cp -r saved_models /content/drive/MyDrive/PCG_Island_Models/